In [ ]:
# !pip install GoogleNews
# !pip install newspaper3k

In [1]:
# Import libraries
import pandas as pd
from GoogleNews import GoogleNews # pip install GoogleNews
from bs4 import BeautifulSoup 
from newspaper import Article # pip install newspaper3k
from newspaper import Config
import matplotlib.pyplot as plt
from urllib.request import urlopen, Request
import time

user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/93.0.4577.82 Safari/537.36'
config = Config()
config.browser_user_agent = user_agent

# Get News from FinViz

https://towardsdatascience.com/stock-news-sentiment-analysis-with-python-193d4b4378d4

In [ ]:
def getFinVizLinks(tickers):
    # Get Data
    finwiz_url = 'https://finviz.com/quote.ashx?t='

    news_tables = {}

    for ticker in tickers:
        url = finwiz_url + ticker
        req = Request(url=url,headers={'user-agent': 'my-app/0.0.1'}) 
        resp = urlopen(req)    
        html = BeautifulSoup(resp, features="lxml")
        news_table = html.find(id='news-table')
        news_tables[ticker] = news_table

    # Iterate through the news
    parsed_news = []
    for file_name, news_table in news_tables.items():
        for x in news_table.findAll('tr'):
            text = x.a.get_text()
            link = x.a.get('href')
            #date_scrape = x.td.text.split()

            #if len(date_scrape) == 1:
            #    time = date_scrape[0]

            #else:
            #    date = date_scrape[0]
            #    time = date_scrape[1]

            ticker = file_name.split('_')[0]

            parsed_news.append([ticker, link])

    #print(parsed_news)
    return(parsed_news)

In [2]:
def getArticleSummary(parsed_news):

    list=[]
    for ind in parsed_news:
        dicti={}
        article = Article(ind[1],config=config)
        try:
             
            article.download()
            article.parse()
            article.nlp()

            dicti['Ticker']=ind[0]
            dicti['Title']=article.title
            dicti['Article']=article.text
            dicti['Summary']=article.summary
            dicti['Link']=ind[1]
            if article.publish_date == None:
                dicti['Date'] = date
            else:
                dicti['Date']= article.publish_date
                date = article.publish_date
            
            list.append(dicti)
        except:
            pass
    news_df=pd.DataFrame(list)
    return(news_df)

In [ ]:
# Parameters 
tickers = ['AAPL']

def getFinViz(tickers):
    parsed_news = getFinVizLinks(tickers)
    news_df = getArticleSummary(parsed_news)
    return(news_df)

In [ ]:
finviz_news = getFinViz(tickers)
finviz_news.to_csv("AAPL_News_finviz.csv")

In [ ]:
tickers = ['TSLA']
finviz_news = getFinViz(tickers)
finviz_news.to_csv("TSLA_News_finviz.csv")

# Get News from Google News

https://medium.com/analytics-vidhya/googlenews-api-live-news-from-google-news-using-python-b50272f0a8f0

In [3]:
def getGoogleNewsLinks(tickers, start_date, end_date):

    parsed_news = []
    
    googlenews=GoogleNews(start = start_date, end = end_date) #month/day/year

    googlenews.search(toSearch)
    
    for i in range(2, 20):
        googlenews.getpage(i)
        result=googlenews.result()
        df=pd.DataFrame(result)
    
    for ind in df.index:
        link = df['link'][ind]
        parsed_news.append([toSearch, link])
    
    return(parsed_news)
#print(df)

In [4]:
toSearch = 'AAPL'
def getGoogleNews(tickers, start, end):
    parsed_news = getGoogleNewsLinks(tickers, start, end)
    news_df = getArticleSummary(parsed_news)
    return(news_df)

In [ ]:
# prevent error429 by working with batches (per month)
time_period = {1: ['01/01/2021', '01/31/2021'],
              2: ['02/01/2021', '02/28/2021'],
              3: ['03/01/2021', '03/31/2021'],
              4: ['04/01/2021', '04/30/2021'],
              5: ['05/01/2021', '05/31/2021'],
              6: ['06/01/2021', '06/30/2021'],
              7: ['07/01/2021', '07/31/2021'],
              8: ['08/01/2021', '08/31/2021'],
              9: ['09/01/2021', '09/30/2021']}


for month_index in range(9, 10):
    start_date, end_date = time_period[month_index][0], time_period[month_index][1]
    google_news = getGoogleNews(toSearch, start_date, end_date)
    google_news
    google_news.to_csv(f"AAPL_News_google_month{month_index}.csv")
    print(f'done: month {month_index}')
#     time.sleep(120) # 2mins interval to prevent error429

In [ ]:
toSearch = 'TSLA'
def getGoogleNews(tickers):
    parsed_news = getGoogleNewsLinks(tickers)
    news_df = getArticleSummary(parsed_news)
    return(news_df)
google_news = getGoogleNews(toSearch)
google_news.to_csv("TSLA_News_google.csv")

In [ ]:
google_news.head()

In [ ]:
finviz_news.head()

# Merging csv files 

In [4]:
import os
import glob
import pandas as pd
os.chdir('D:/Programming/Jupyter\IS453 - Financial Analytics/Github/News/AAPL_News_Google') #folder path

In [6]:
extension = 'csv'
all_filenames = [i for i in glob.glob('*.{}'.format(extension))]


#combine all files in the list
combined_csv = pd.concat([pd.read_csv(f) for f in all_filenames])

#export csv
combined_csv.to_csv('combined_aapl_googlenews.csv', index=False, encoding = 'utf-8-sig')

In [ ]:
combined_csv.head()